# SONICS training — explainable AI-music detectionRuns milestones **M3–M6** on Kaggle, where SONICS attaches without consuminglocal disk. The laptop this repo was built on has 4 GB VRAM and cannot hold thedataset, so this notebook is where reportable numbers come from.**Before running:** Settings → Accelerator = **GPU T4**, Internet = **On**(needed to fetch the MERT checkpoint), and add the dataset `awsaf49/sonics-dataset`.The GPU quota (~30 h/week) is the binding constraint. Settle the architecture ona subset first — `LIMIT_SONGS` below — before launching the full run or theablation sweep.

In [ ]:
# The image already ships torch/torchaudio/transformers -- do NOT reinstall them.# These are the pieces it lacks. MP3 and reverb are required, not optional:# MP3 is the base paper's headline robustness condition.!pip install -q nnAudio audiomentations fast_mp3_augment pyroomacoustics!pip install -q git+https://github.com/Laksh-D25/explainable-aimd.git

In [ ]:
import os, sys, glob, jsonfrom pathlib import Pathimport numpy as np, pandas as pd, torchimport aimdfrom aimd.pipeline import (TrainConfig, build_model, calibrate_and_threshold,                           load_checkpoint, resolve_device, train_model)DEVICE = resolve_device("auto")print("aimd", aimd.__file__)print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")assert DEVICE.startswith("cuda"), "Enable the GPU accelerator before running."

## 1. Locate the dataThe Kaggle mirror's layout is discovered rather than assumed — it has changedbetween versions, and a wrong hard-coded path fails much later and less clearly.

In [ ]:
ROOT = Path("/kaggle/input")print("available datasets:", [p.name for p in ROOT.iterdir()] if ROOT.exists() else "none")csvs = {Path(p).name: p for p in glob.glob(str(ROOT / "**" / "*.csv"), recursive=True)}print("\ncsvs found:")for name, path in sorted(csvs.items()):    print(f"  {name:20s} {path}")audio = glob.glob(str(ROOT / "**" / "*.mp3"), recursive=True)[:3]print("\nsample audio:", audio if audio else "NONE FOUND -- check the dataset is attached")AUDIO_ROOT = Path(audio[0]).parent.parent if audio else Noneprint("audio root guess:", AUDIO_ROOT)

## 2. Build the manifestSONICS publishes its own train/valid/test splits. Those are adopted verbatim:a freshly generated partition, however carefully stratified, measures adifferent problem and would break comparability with SONICS's published ~0.97 F1.If the official split files are absent, we fall back to a leak-free split bysong id — and say so, because the comparison then carries a caveat.

In [ ]:
from aimd.data.manifest import (apply_official_splits, load_sonics_manifest,                                split_by_song, summarize, assert_no_leakage)manifest = load_sonics_manifest(csvs["real_songs.csv"], csvs["fake_songs.csv"])print(f"{len(manifest)} songs, taxonomy: {manifest['taxonomy'].value_counts().to_dict()}")# Resolve audio paths against the discovered root.def resolve(p):    hits = glob.glob(str(ROOT / "**" / Path(str(p)).name), recursive=True)    return hits[0] if hits else Noneif not Path(str(manifest['path'].iloc[0])).exists():    print("resolving audio paths (slow, one pass)...")    index = {Path(p).stem: p for p in glob.glob(str(ROOT / "**" / "*.mp3"), recursive=True)}    manifest["path"] = manifest["song_id"].astype(str).map(index)    missing = manifest["path"].isna().sum()    print(f"  unresolved: {missing}")    manifest = manifest.dropna(subset=["path"])official = {k: csvs[f] for k, f in [("train","train.csv"),("val","valid.csv"),("test","test.csv")] if f in csvs}if len(official) == 3:    manifest = apply_official_splits(manifest, official)    print("\nusing SONICS official splits")else:    manifest = split_by_song(manifest, seed=1337)    print("\nWARNING: official splits not found; generated our own."          " In-distribution numbers are NOT directly comparable to the SONICS paper.")assert_no_leakage(manifest)manifest.to_csv("/kaggle/working/manifest.csv", index=False)print(summarize(manifest).to_string())

## 3. Sanity gate before spending quotaOverfit a small subset. If the head cannot separate ~100 songs it has seenrepeatedly, the data path is broken and the full run would waste hours.

In [ ]:
LIMIT_SONGS = 100gate_cfg = TrainConfig(epochs=8, batch_size=8, clip_seconds=10.0, clips_per_song=4,                       augment=False, limit_songs=LIMIT_SONGS, num_workers=2, amp=True)gate_model = build_model(DEVICE)print(f"trainable: {sum(p.numel() for p in gate_model.trainable_parameters()):,}")gate = train_model(gate_model, manifest, gate_cfg, device=DEVICE,                   checkpoint_dir=Path("/kaggle/working/gate"))final_loss = gate["history"]["train_loss"].iloc[-1]print(f"\nfinal train loss {final_loss:.4f}  best val F1 {gate['best']['f1']:.4f}")assert final_loss < gate["history"]["train_loss"].iloc[0], "did not learn -- STOP and debug"

## 4. Full training runSet `EPOCHS` against your remaining quota. Augmentation is on and applies to thetrain split only; the held-out robustness conditions in section 6 were neverseen, so those results measure generalisation rather than memorisation.

In [ ]:
CONFIG = TrainConfig(    epochs=15, batch_size=8, lr=1e-3, weight_decay=0.01, aux_loss_weight=0.3,    clip_seconds=10.0, clips_per_song=8, max_clips=12,    augment=True, num_workers=2, amp=True, seed=1337,)model = build_model(DEVICE)result = train_model(model, manifest, CONFIG, device=DEVICE,                     checkpoint_dir=Path("/kaggle/working/run"))result["history"].to_csv("/kaggle/working/history.csv", index=False)result["history"]

## 5. CalibrateTemperature is fitted on validation, never on test. The operating threshold ischosen against a 1% false-positive budget rather than left at 0.5: wronglyflagging a human artist's track is not symmetric with missing an AI one.

In [ ]:
scaler, threshold, calibration = calibrate_and_threshold(    model, manifest, CONFIG.spec(), target_fpr=0.01, device=DEVICE, amp=True)print(json.dumps(calibration, indent=2))print(f"\nECE {calibration['ece_before']:.4f} -> {calibration['ece_after']:.4f}")

## 6b. FakeMusicCaps — the cross-generator manifestThe 12 GB release (Zenodo 15063698) holds **only generated audio**: 27,605 clipsof 10 s / 16 kHz mono across five TTM models, one directory per model, thedirectory name being the attribution label.The real class is a **separate MusicCaps download**. Supply it — and process itidentically. Pairing these clips with real songs from SONICS instead would makethe detector separate 10 s of 16 kHz audio from full-length YouTube tracks,which measures the domain gap, not the generator. That number would look strongand mean nothing.

In [ ]:
from aimd.data.manifest import load_fakemusiccaps_manifestFMC_ROOT = None       # e.g. "/kaggle/input/fakemusiccaps/FakeMusicCaps"FMC_REAL_ROOT = None  # MusicCaps originals, resampled the same wayif FMC_ROOT:    fmc = load_fakemusiccaps_manifest(FMC_ROOT, FMC_REAL_ROOT)    fmc.to_csv("/kaggle/working/fmc_manifest.csv", index=False)    print(fmc.groupby(["taxonomy", "source"], dropna=False).size().to_string())    if not (fmc["label"] == 0).any():        print("\nNO REAL CLASS -- detection metrics cannot be computed; "              "attribution only. Add MusicCaps originals via FMC_REAL_ROOT.")    FMC_MANIFEST = "/kaggle/working/fmc_manifest.csv"else:    FMC_MANIFEST = None    print("FMC_ROOT not set -- the headline cross-generator number will be skipped")

## 6. The three protocols`in_distribution` is a sanity number against SONICS's ~0.97. The **cross-generator**row is the headline result, set against the base paper's F1 **0.629** — and itrequires a FakeMusicCaps manifest, so add that dataset to run it.

In [ ]:
from aimd.eval.protocols import (evaluate_cross_generator, evaluate_in_distribution,                                 evaluate_robustness, results_table)results = [evaluate_in_distribution(model, manifest, CONFIG.spec(), scaler, threshold,                                    device=DEVICE, amp=True)]# FMC_MANIFEST comes from section 6b.if FMC_MANIFEST:    cross = evaluate_cross_generator(model, pd.read_csv(FMC_MANIFEST), CONFIG.spec(),                                     scaler, threshold, device=DEVICE, amp=True)    results.append(cross)    print(f"cross-generator F1 {cross.metrics['f1']:.4f} "          f"vs base paper {cross.extra['baseline_f1']} "          f"({cross.extra['delta_vs_baseline']:+.4f})")else:    print("no FakeMusicCaps manifest -- the headline cross-generator number was NOT produced")table = results_table(results)table.to_csv("/kaggle/working/protocols.csv")table

In [ ]:
robustness = evaluate_robustness(model, manifest, CONFIG.spec(), scaler, threshold,                                 device=DEVICE, amp=True)robustness.to_csv("/kaggle/working/robustness.csv")robustness[["f1", "f1_drop", "ece", "family"]]

## 7. AblationsEach run isolates one architectural choice. This is what justifies the designempirically instead of by assertion — and it is the most quota-hungry step, sorun it only once the architecture is settled.

In [ ]:
ABLATIONS = {    "baseline":       {},    "pool_mean":      {"frame_pool": "mean"},    "layers_last":    {"layer_mode": "last"},    "song_mean":      {"song_pool": "mean"},    "self_attn_on":   {"use_self_attn": True},}rows = []for name, overrides in ABLATIONS.items():    print(f"\n=== {name} ===")    m = build_model(DEVICE, **overrides)    train_model(m, manifest, CONFIG, device=DEVICE, checkpoint_dir=Path(f"/kaggle/working/abl_{name}"))    s, t, _ = calibrate_and_threshold(m, manifest, CONFIG.spec(), device=DEVICE, amp=True)    r = evaluate_in_distribution(m, manifest, CONFIG.spec(), s, t, device=DEVICE, amp=True)    rows.append({"ablation": name, **r.metrics})    del m; torch.cuda.empty_cache()ablation_table = pd.DataFrame(rows).set_index("ablation")ablation_table.to_csv("/kaggle/working/ablations.csv")ablation_table

## 8. ExplanationsFaithfulness is checked before any explanation figure is trusted: masking theregions an explanation calls important must move the logit more than maskingrandom regions. An explanation that fails this is reported as unfaithful, notquietly dropped.Note that gradient-based relevance is uninformative on a *saturated* model —if training loss collapsed to ~0, these numbers mean nothing.

In [ ]:
from aimd.data.datasets import SongClipsDatasetfrom aimd.eval.protocols import rows_for_splitfrom aimd.xai.faithfulness import deletion_curvefrom aimd.xai.relevance import temporal_relevancefrom aimd.xai.shap_layers import agreement_with_attention, layer_shapleytest_rows = rows_for_split(manifest, "test")ds = SongClipsDataset(test_rows.head(20), CONFIG.spec(), split="test")model.eval()audit = []for i in range(min(10, len(ds))):    item = ds[i]    wav = item["clips"].unsqueeze(0).to(DEVICE)    mask = item["clip_mask"].unsqueeze(0).to(DEVICE)    rel = temporal_relevance(model, wav, mask)    hidden = model.backbone(wav.flatten(0, 1), no_grad=True)    phi = layer_shapley(model, hidden.view(*wav.shape[:2], *hidden.shape[1:]), mask)    curve = deletion_curve(model, wav, rel, mask, steps=8)    audit.append({        "song_id": item["song_id"],        "faithfulness_gap": curve.gap,        "beats_chance": curve.gap < 0,        "l1_l3_agreement": agreement_with_attention(phi, model.layer_attn.logits.softmax(0)),    })audit = pd.DataFrame(audit)audit.to_csv("/kaggle/working/faithfulness.csv", index=False)print(f"explanations beating chance: {audit['beats_chance'].mean():.0%}")print(f"mean L1-vs-L3 agreement: {audit['l1_l3_agreement'].mean():+.3f}")audit

## 8b. FiguresFour figures for the write-up, each carrying one of the project's four claims.Every figure writes its numbers to CSV beside it — that table is both theaccessible view and the appendix material. PDFs are vector, for print.

In [ ]:
from aimd.eval.report import (faithfulness_curves, layer_profile, reliability_diagram,                              results_markdown, robustness_chart)FIGS = Path("/kaggle/working/figures")val = predict(model, build_loader(manifest, CONFIG.spec(), "val"), device=DEVICE, amp=True)reliability_diagram(val["label"], val["prob"], scaler.transform(val["logit"]), FIGS/"calibration")robustness_chart(robustness, FIGS/"robustness")# One representative song for the explanation figures.item = ds[0]w = item["clips"].unsqueeze(0).to(DEVICE); mk = item["clip_mask"].unsqueeze(0).to(DEVICE)rel = temporal_relevance(model, w, mk)hh = model.backbone(w.flatten(0, 1), no_grad=True)phi = layer_shapley(model, hh.view(*w.shape[:2], *hh.shape[1:]), mk)layer_profile(model.layer_attn.logits.softmax(0).detach().cpu().numpy(), phi, FIGS/"layers")faithfulness_curves(deletion_curve(model, w, rel, mk, steps=8), FIGS/"faithfulness")print(results_markdown(table))print("\nfigures:", sorted(f.name for f in FIGS.glob("*.pdf")))

## 9. Collect outputsEverything written to `/kaggle/working/` persists as notebook output. Download`protocols.csv`, `robustness.csv`, `ablations.csv` and `faithfulness.csv` — thosefour are the report's tables.

In [ ]:
for f in sorted(Path("/kaggle/working").glob("*.csv")):    print(f"{f.name:24s} {f.stat().st_size:>8,} bytes")